Dado 3: Localização Georreferenciada dos Acidentes

* Fonte Oficial: Portal de Dados Abertos de Porto Alegre (DataPOA / EPTC).
* Entrada: `dados_brutos/bruto_eptc_acidentes.csv` e `dados_brutos/bruto_osm_malha_poa.geojson`
* Tratamento e Critério de Precisão Espacial:
  1. Coordenadas Nativas: Filtra acidentes com ciclistas (`bicicleta > 0`) e converte os **1.458 registros** com GPS original da EPTC para o sistema oficial `EPSG:4326` (`EPTC_Original`).
  2. Recuperação Geométrica Local (Offline): Dos 169 registros sem GPS nativo, aplicou-se normalização textual, correspondência aproximada (*fuzzy matching*) e cruzamento topológico direto sobre a malha viária local do OpenStreetMap:
     * 62 pontos recuperados pela interseção exata de esquinas (`log1` ∩ `log2`, identificados como `Cruzamento_Local_OSM`).
     * 29 pontos recuperados em vias locais/coletoras curtas ($\le 3\text{ km}$) com numeração predial informada (`Via_Predial_Curta_OSM`).
  3. Descarte por Rigor Espacial: Os **78 registros restantes** (37 sem número/esquina e 41 em avenidas extensas sem cruzamento mapeado) foram descartados intencionalmente deste arquivo espacial para evitar viés de centroide geométrico no modelo — permanecendo preservados nos arquivos `limpo_01` e `limpo_02` para análises estatísticas globais.
* Saída: `dados_limpos/limpo_03_geolocalizacao_acidentes.geojson` (**1.549 pontos** — **95,2% de cobertura**).

#### Etapa 1: Diagnóstico de Completude e Precisão dos Registros sem GPS

* **Por que este diagnóstico foi feito?** Ao inspecionar a base bruta da EPTC, constatou-se que **169 dos 1.627 acidentes com ciclistas (10,4%)** não possuíam coordenadas geográficas (`latitude`/`longitude`) preenchidas. Antes de tentar recuperar esses pontos por endereço textual, foi necessário auditar a qualidade da escrita da EPTC e verificar quantos registros possuíam localização exata (número predial ou cruzamento).
* **No que isso influencia o projeto?** Se geocodificássemos registros que contêm apenas o nome de avenidas extensas (sem número nem esquina), o algoritmo posicionaria o acidente no centro geométrico da via (centroide), gerando erros espaciais de centenas de metros e criando falsos pontos críticos de acidentes no modelo de Machine Learning.
* **Resultado e Decisão Tomada:** O diagnóstico abaixo comprovou que **132 registros (78,1%)** possuem endereço de alta precisão (passíveis de recuperação local) e **37 registros (21,3%)** possuem apenas o nome da via ou nenhum endereço — fundamentando a decisão do grupo de recuperar seletivamente os casos precisos e descartar os ambíguos deste arquivo espacial.


In [7]:
import pandas as pd

df_diag = pd.read_csv("../../data/1_raw/bruto_eptc_acidentes.csv", sep=";", low_memory=False)
df_diag.columns = df_diag.columns.str.lower().str.strip()

df_bike = df_diag[df_diag['bicicleta'] > 0].copy()
lat = pd.to_numeric(df_bike['latitude'].astype(str).str.replace(',', '.'), errors='coerce')
sem_gps = df_bike[lat.isna() | (lat == 0)].copy()

tem_rua1 = sem_gps['log1'].notna() & (~sem_gps['log1'].astype(str).isin(['nan', 'None', '']))
tem_predial = pd.to_numeric(sem_gps['predial1'], errors='coerce').fillna(0) > 0
tem_rua2 = sem_gps['log2'].notna() & (~sem_gps['log2'].astype(str).isin(['nan', 'None', '']))

precisao_alta = sem_gps[tem_rua1 & (tem_predial | tem_rua2)]
so_nome_rua = sem_gps[tem_rua1 & (~tem_predial) & (~tem_rua2)]

print(f"Total de registros sem GPS analisados: {len(sem_gps)}")
print("-" * 65)
print(f"• Com endereço exato (Rua + Número OU Esquina): {len(precisao_alta)} ({len(precisao_alta)/len(sem_gps)*100:.1f}%)")
print(f"• Apenas com o nome da rua (Risco de centroide): {len(so_nome_rua)} ({len(so_nome_rua)/len(sem_gps)*100:.1f}%)")
print(f"• Sem endereço algum                           : {len(sem_gps) - len(precisao_alta) - len(so_nome_rua)}")

Total de registros sem GPS analisados: 169
-----------------------------------------------------------------
• Com endereço exato (Rua + Número OU Esquina): 132 (78.1%)
• Apenas com o nome da rua (Risco de centroide): 36 (21.3%)
• Sem endereço algum                           : 1


#### Etapa 2: Recuperação Geométrica Local e Geração do Arquivo Limpo

In [8]:
import re
import difflib
import unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd

# 1. Carrega acidentes da EPTC e o mapa local de ruas
df = pd.read_csv("../../data/1_raw/bruto_eptc_acidentes.csv", sep=";", low_memory=False)
df.columns = df.columns.str.lower().str.strip()
ruas_osm = gpd.read_file("../../data/1_raw/bruto_osm_malha_poa.geojson")

df_ciclistas = df[df['bicicleta'] > 0].copy()
df_ciclistas['latitude'] = pd.to_numeric(df_ciclistas['latitude'].astype(str).str.replace(',', '.'), errors='coerce')
df_ciclistas['longitude'] = pd.to_numeric(df_ciclistas['longitude'].astype(str).str.replace(',', '.'), errors='coerce')

def normalizar_via(texto):
    if isinstance(texto, (list, tuple, np.ndarray)):
        texto = texto[0] if len(texto) > 0 else ""
    if texto is None or (isinstance(texto, float) and pd.isna(texto)):
        return ""
    
    t = unicodedata.normalize('NFKD', str(texto)).encode('ASCII', 'ignore').decode('utf-8').upper().strip()
    t = re.sub(r'[^A-Z0-9\s]', ' ', t)
    t = re.sub(r'^AV\b\s*', 'AVENIDA ', t)
    t = re.sub(r'^R\b\s*', 'RUA ', t)
    t = re.sub(r'^ESTR\b\s*', 'ESTRADA ', t)
    t = re.sub(r'^TV\b\s*', 'TRAVESSA ', t)
    t = re.sub(r'\bGEN\b', 'GENERAL', t)
    t = re.sub(r'\bDR\b', 'DOUTOR', t)
    t = re.sub(r'\bPROF\b', 'PROFESSOR', t)
    t = re.sub(r'\bENG\b', 'ENGENHEIRO', t)
    t = re.sub(r'\bCEL\b', 'CORONEL', t)
    return re.sub(r'\s+', ' ', t).strip()

# Prepara dicionário local de geometrias das ruas do OSM
ruas_osm['nome_norm'] = ruas_osm['name'].apply(normalizar_via)
ruas_validas = ruas_osm[~ruas_osm['nome_norm'].isin(['', 'NONE', 'NAN'])]
geometria_por_rua = ruas_validas.dissolve(by='nome_norm')['geometry'].to_dict()
lista_nomes_osm = list(geometria_por_rua.keys())

def encontrar_via_osm(nome_normalizado):
    if not nome_normalizado:
        return None
    if nome_normalizado in geometria_por_rua:
        return geometria_por_rua[nome_normalizado]
    parecidos = difflib.get_close_matches(nome_normalizado, lista_nomes_osm, n=1, cutoff=0.85)
    if parecidos:
        return geometria_por_rua[parecidos[0]]
    return None

# Filtra os 132 elegíveis (sem GPS, mas com número predial > 0 OU rua transversal)
sem_gps = df_ciclistas['latitude'].isna() | (df_ciclistas['latitude'] == 0)
tem_rua1 = df_ciclistas['log1'].notna() & (~df_ciclistas['log1'].astype(str).isin(['nan', 'None', '']))
tem_predial = pd.to_numeric(df_ciclistas['predial1'], errors='coerce').fillna(0) > 0
tem_rua2 = df_ciclistas['log2'].notna() & (~df_ciclistas['log2'].astype(str).isin(['nan', 'None', '']))

elegiveis = sem_gps & tem_rua1 & (tem_predial | tem_rua2)
df_ciclistas['origem_gps'] = 'EPTC_Original'

for idx, row in df_ciclistas[elegiveis].iterrows():
    r1 = normalizar_via(row['log1'])
    r2 = normalizar_via(row['log2'])
    predial = pd.to_numeric(row['predial1'], errors='coerce')
    
    geom1 = encontrar_via_osm(r1)
    geom2 = encontrar_via_osm(r2)
    
    # Tentativa 1: Cruzamento exato (interseção entre via 1 e via 2)
    if geom1 is not None and geom2 is not None:
        intersecao = geom1.intersection(geom2)
        if not intersecao.is_empty:
            ponto = intersecao.centroid
            df_ciclistas.at[idx, 'latitude'] = ponto.y
            df_ciclistas.at[idx, 'longitude'] = ponto.x
            df_ciclistas.at[idx, 'origem_gps'] = 'Cruzamento_Local_OSM'
            continue
            
    # Tentativa 2: Via com número predial (usa ponto representativo da via quando não é avenida longa demais)
    if geom1 is not None and pd.notna(predial) and predial > 0:
        comprimento_km = geom1.length * 111
        if comprimento_km <= 3.0:
            ponto = geom1.interpolate(0.5, normalized=True)
            df_ciclistas.at[idx, 'latitude'] = ponto.y
            df_ciclistas.at[idx, 'longitude'] = ponto.x
            df_ciclistas.at[idx, 'origem_gps'] = 'Via_Predial_Curta_OSM'

df_valido = df_ciclistas.dropna(subset=['latitude', 'longitude']).copy()
df_valido = df_valido[(df_valido['latitude'] != 0) & (df_valido['longitude'] != 0)]

colunas_id = [c for c in ['id', 'data', 'latitude', 'longitude', 'origem_gps'] if c in df_valido.columns]
gdf = gpd.GeoDataFrame(
    df_valido[colunas_id],
    geometry=gpd.points_from_xy(df_valido['longitude'], df_valido['latitude']),
    crs="EPSG:4326"
)

gdf.to_file("../../data/3_done/limpo_03_geolocalizacao_acidentes.geojson", driver="GeoJSON")
print(f"Concluído! Total final georreferenciado: {len(gdf)} de {len(df_ciclistas)} acidentes ({len(gdf)/len(df_ciclistas)*100:.1f}%).")
print("\nDetalhamento por origem:")
print(gdf['origem_gps'].value_counts())

/Users/aluno-53/Documents/RIA26_Mobilidade_Urbana /seguranca_viaria_poa/venv/lib/python3.12/site-packages/geopandas/io/file.py:582: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


Concluído! Total final georreferenciado: 1549 de 1627 acidentes (95.2%).

Detalhamento por origem:
origem_gps
EPTC_Original            1458
Cruzamento_Local_OSM       62
Via_Predial_Curta_OSM      29
Name: count, dtype: int64
